# 📤 Export Data for Frontend Dashboard

This notebook exports processed results to JSON format for the web dashboard.

**Exports:**
- Area statistics by year
- Trend data (2020-2025)
- Ecological summaries
- Classification maps

In [1]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

# Setup paths
frontend_data = Path('../FRONTEND/data')
frontend_data.mkdir(exist_ok=True)

print("✅ Export environment ready")
print(f"📁 Output folder: {frontend_data.absolute()}")

✅ Export environment ready
📁 Output folder: c:\Users\HP LAPTOP 15s\matlabTiff\notebooks\..\FRONTEND\data


## 1. Calculate Area Statistics

Calculate km² for each class based on pixel counts and resolution.

In [2]:
# Mock data for demonstration - replace with actual model predictions
# Pixel resolution: 10m x 10m = 100 m² per pixel
# 1 km² = 1,000,000 m² = 10,000 pixels

PIXEL_TO_KM2 = 100 / 1_000_000  # 100 m² per pixel to km²

# Sample pixel counts by year (replace with actual predictions)
year_data = {
    2020: {'Seagrass': 4201000, 'Sand': 1220000, 'Water': 8700000, 'Landmass': 780000},
    2021: {'Seagrass': 4300000, 'Sand': 1210000, 'Water': 8750000, 'Landmass': 781000},
    2022: {'Seagrass': 4400000, 'Sand': 1208000, 'Water': 8800000, 'Landmass': 782000},
    2023: {'Seagrass': 4502000, 'Sand': 1205000, 'Water': 8903000, 'Landmass': 781000},
    2024: {'Seagrass': 4550000, 'Sand': 1200000, 'Water': 8920000, 'Landmass': 781000},
    2025: {'Seagrass': 4600000, 'Sand': 1195000, 'Water': 8950000, 'Landmass': 781000}
}

def calculate_areas(pixel_counts):
    """Convert pixel counts to km²"""
    return {cls: round(count * PIXEL_TO_KM2, 1) for cls, count in pixel_counts.items()}

def calculate_change(current, previous):
    """Calculate percentage change"""
    if previous == 0:
        return 0.0
    return round(((current - previous) / previous) * 100, 1)

# Process each year
area_stats = {}
years = sorted(year_data.keys())

for i, year in enumerate(years):
    areas = calculate_areas(year_data[year])
    stats = []
    
    for category, area in areas.items():
        # Calculate change from previous year
        if i > 0:
            prev_year = years[i-1]
            prev_area = calculate_areas(year_data[prev_year])[category]
            change = calculate_change(area, prev_area)
        else:
            change = 0.0
        
        stats.append({
            'category': category,
            'area': area,
            'change': change
        })
    
    area_stats[str(year)] = stats

print("✅ Area statistics calculated")
print(f"Years: {years}")

✅ Area statistics calculated
Years: [2020, 2021, 2022, 2023, 2024, 2025]


## 2. Generate Trend Data

Prepare time-series data for charts.

In [3]:
# Extract trend data for line chart
trend_data = {
    'years': years,
    'seagrass': [],
    'sand': [],
    'water': [],
    'landmass': []
}

for year in years:
    areas = calculate_areas(year_data[year])
    trend_data['seagrass'].append(areas['Seagrass'])
    trend_data['sand'].append(areas['Sand'])
    trend_data['water'].append(areas['Water'])
    trend_data['landmass'].append(areas['Landmass'])

print("✅ Trend data prepared")

✅ Trend data prepared


## 3. Create Ecological Summaries

Generate text summaries for each year.

In [4]:
# Generate summaries based on trends
eco_summaries = {}

for i, year in enumerate(years):
    if i == 0:
        eco_summaries[str(year)] = f"Baseline year established for coastal monitoring. Initial seagrass coverage: {trend_data['seagrass'][i]} km²."
    else:
        sg_change = trend_data['seagrass'][i] - trend_data['seagrass'][i-1]
        sand_change = trend_data['sand'][i] - trend_data['sand'][i-1]
        
        sg_status = "increased" if sg_change > 0 else "decreased" if sg_change < 0 else "remained stable"
        sand_status = "expanded" if sand_change > 0 else "contracted" if sand_change < 0 else "remained stable"
        
        eco_summaries[str(year)] = f"""During {year}, seagrass coverage <b>{sg_status}</b> by {abs(sg_change):.1f} km² compared to {year-1}. 
        Sand shoals {sand_status} by {abs(sand_change):.1f} km². 
        Overall, the coastal ecosystem shows {'positive' if sg_change > 0 else 'concerning'} trends in biodiversity indicators."""

print("✅ Ecological summaries generated")

✅ Ecological summaries generated


## 4. Export to JSON Files

Save all data as JSON for frontend consumption.

In [5]:
# Export area statistics
with open(frontend_data / 'area_stats.json', 'w') as f:
    json.dump(area_stats, f, indent=2)
print("✅ Exported: area_stats.json")

# Export trend data
with open(frontend_data / 'trend_data.json', 'w') as f:
    json.dump(trend_data, f, indent=2)
print("✅ Exported: trend_data.json")

# Export ecological summaries
with open(frontend_data / 'eco_summaries.json', 'w') as f:
    json.dump(eco_summaries, f, indent=2)
print("✅ Exported: eco_summaries.json")

print("\n🎉 All data exported to FRONTEND/data/")

✅ Exported: area_stats.json
✅ Exported: trend_data.json
✅ Exported: eco_summaries.json

🎉 All data exported to FRONTEND/data/


## 5. Export Satellite True Color Images

Convert True Color satellite images to PNG format for web display.

In [7]:
import rasterio
import matplotlib.pyplot as plt
from PIL import Image
import shutil
import glob

# Export satellite images for available years
available_years = [2020, 2021, 2022, 2023, 2024, 2025]

for year in available_years:
    print(f"\n📷 Processing year {year}...")
    
    # Strategy 1: Look for JPG True Color (easiest)
    jpg_pattern = f'../data/years/{year}/*True_color.jpg'
    jpg_matches = glob.glob(jpg_pattern)
    
    if jpg_matches:
        # Just copy the JPG directly
        src_path = jpg_matches[0]
        dst_path = frontend_data / f'map_{year}.png'
        shutil.copy(src_path, dst_path)
        print(f"   ✅ Copied JPG for {year}")
        continue
    
    # Strategy 2: Look for TIFF True Color and convert
    tiff_pattern = f'../data/years/{year}/*True_color.tif*'
    tiff_matches = glob.glob(tiff_pattern)
    
    if tiff_matches:
        src_path = tiff_matches[0]
        print(f"   🔄 Converting TIFF for {year}...")
        
        try:
            with rasterio.open(src_path) as src:
                # Read all bands
                data = src.read()
                
                # Handle different TIFF formats
                if data.shape[0] >= 3:
                    # Take first 3 bands (RGB)
                    rgb = data[:3]
                    
                    # Transpose to (H, W, 3) for image display
                    rgb = np.transpose(rgb, (1, 2, 0))
                    
                    # Normalize to 0-255 range
                    # Use percentile clipping for better contrast
                    for i in range(3):
                        band = rgb[:, :, i].astype(float)
                        # Clip outliers
                        p2, p98 = np.percentile(band[band > 0], [2, 98])
                        band = np.clip(band, p2, p98)
                        # Scale to 0-255
                        band = ((band - p2) / (p98 - p2) * 255)
                        rgb[:, :, i] = band
                    
                    # Convert to uint8
                    rgb = np.clip(rgb, 0, 255).astype(np.uint8)
                    
                    # Save as PNG
                    dst_path = frontend_data / f'map_{year}.png'
                    Image.fromarray(rgb).save(dst_path)
                    print(f"   ✅ Converted TIFF to PNG for {year}")
                else:
                    print(f"   ⚠️  Not enough bands in TIFF for {year}")
        except Exception as e:
            print(f"   ❌ Error processing {year}: {e}")
    else:
        print(f"   ⚠️  No True Color image found for {year}")

print("\n✅ Image export complete!")
print(f"   Exported maps saved to: {frontend_data.absolute()}")
print("\n💡 Tip: For best results, download True_color.jpg versions from Sentinel Hub")


📷 Processing year 2020...
   ✅ Copied JPG for 2020

📷 Processing year 2021...
   ✅ Copied JPG for 2021

📷 Processing year 2022...
   ✅ Copied JPG for 2022

📷 Processing year 2023...
   ✅ Copied JPG for 2023

📷 Processing year 2024...
   ✅ Copied JPG for 2024

📷 Processing year 2025...
   ✅ Copied JPG for 2025

✅ Image export complete!
   Exported maps saved to: c:\Users\HP LAPTOP 15s\matlabTiff\notebooks\..\FRONTEND\data

💡 Tip: For best results, download True_color.jpg versions from Sentinel Hub


## ✅ Export Complete

Frontend can now load data from:
- `FRONTEND/data/area_stats.json`
- `FRONTEND/data/trend_data.json`
- `FRONTEND/data/eco_summaries.json`
- `FRONTEND/data/map_YYYY.png` (Satellite True Color images)

**Next Steps:**
1. Refresh your browser (Ctrl+F5 or Cmd+Shift+R) to clear cache
2. Year-specific satellite maps will display automatically
3. Replace mock data in cell 4 with real predictions from notebook 04 when ready

**Troubleshooting:**
- If maps show black: Download JPG versions of True Color images from Sentinel Hub
- If maps don't load: Check browser console (F12) for errors
- Alternative: Export classification maps from notebook 04's results folder